In [1]:
import os

In [2]:
num_cores = os.cpu_count()
print(num_cores)

8


In [3]:
import os
import pandas as pd
import polars as pl
import numpy as np
from timeit import Timer

In [4]:
os.environ['POLARS_MAX_THREADS'] = '8'

In [5]:
pl.thread_pool_size()

8

In [6]:
def generate_data(number_of_rows):
    rng = np.random.default_rng()

    return {
        "order_id": range(1, number_of_rows + 1),
        "region": rng.choice(
            ["North", "South", "East", "West"], size=number_of_rows
        ),
        "sales_person": rng.choice(
            ["Armstrong", "Aldrin", "Collins"], size=number_of_rows
        ),
        "product": rng.choice(
            ["Helmet", "Oxygen", "Boots", "Gloves"], size=number_of_rows
        ),
        "sales_income": np.ones(number_of_rows),
    }

In [7]:
def create_pandas_dataframe(test_data):
    return pd.DataFrame(test_data).convert_dtypes(dtype_backend="pyarrow")

def create_polars_dataframe(test_data):
    return pl.DataFrame(test_data)

def create_polars_lazyframe(test_data):
    return pl.LazyFrame(test_data)

In [8]:
display_data = generate_data(10)
create_pandas_dataframe(display_data)

,order_id,region,sales_person,product,sales_income
0,1,East,Armstrong,Oxygen,1
1,2,North,Armstrong,Helmet,1
2,3,North,Aldrin,Oxygen,1
3,4,East,Collins,Oxygen,1
4,5,South,Aldrin,Gloves,1
5,6,East,Armstrong,Boots,1
6,7,East,Aldrin,Boots,1
7,8,West,Aldrin,Oxygen,1
8,9,North,Aldrin,Oxygen,1
9,10,East,Armstrong,Helmet,1


In [9]:
def groupby_pandas_dataframe(pandas_df):
    return pandas_df.groupby(["region", "product", "sales_person"])[
        "sales_income"
    ].sum()

def groupby_polars_dataframe(polars_df):
    return polars_df.group_by(["region", "product", "sales_person"]).agg(
        total_sales=pl.col("sales_income").sum()
    )

def groupby_polars_lazyframe(polars_lf):
    return polars_lf.group_by(["region", "product", "sales_person"]).agg(
        total_sales=pl.col("sales_income").sum()
    ).collect()

In [10]:
test_data = generate_data(number_of_rows=1000000) # max on CPU: 10000000

# Timing for GroupBy operations
# pandas
pandas_df = create_pandas_dataframe(test_data)
size_bytes = pandas_df.memory_usage(index=True).sum()
size_mb = size_bytes / (1024 ** 2)
print(f"Pandas DataFrame size: {size_mb:.2f} MB")

# polars eager dataframe
polars_df = create_polars_dataframe(test_data)
size_bytes = polars_df.estimated_size()
size_mb = size_bytes / (1024 ** 2)
print(f"Polars Eager DataFrame size: {size_mb:.2f} MB")

# polars lazyframe 
polars_lf = create_polars_lazyframe(test_data)


Pandas DataFrame size: 43.47 MB
Polars Eager DataFrame size: 32.03 MB


In [16]:
pandas_timer = Timer(lambda: groupby_pandas_dataframe(pandas_df))
print(f"Pandas Analysis Time: {pandas_timer.timeit(number=100):.6f} seconds")

polars_timer = Timer(lambda: groupby_polars_dataframe(polars_df))
print(f"Polars Analysis Time: {polars_timer.timeit(number=100):.6f} seconds")

polars_lazyframe_timer = Timer(lambda: groupby_polars_lazyframe(polars_lf))
print(f"Polars LazyFrame Analysis Time: {polars_lazyframe_timer.timeit(number=100):.6f} seconds")

Pandas Analysis Time: 6.455103 seconds
Polars Analysis Time: 1.329943 seconds
Polars LazyFrame Analysis Time: 1.310174 seconds
